# 08_final_run — Jalankan ulang semua analisis pada corpus final (119 dokumen)
Prasyarat: `03_clean/screening.xlsx` sudah diganti dengan versi terbaru (I = 119).
Jalankan **Runtime → Run all**. Semua hasil berawalan `final_`; file lama tidak ditimpa, kecuali file corpus dan `M.rds`.

## Sel 1 — SEL PEMBUKA (Python)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%load_ext rpy2.ipython
!apt-get -qq update > /dev/null
!apt-get -qq install -y libpoppler-cpp-dev libglpk-dev libxml2-dev libcurl4-openssl-dev libssl-dev libfontconfig1-dev libharfbuzz-dev libfribidi-dev libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev > /dev/null
print('Sel 1 selesai')

## Sel 2 — SEL PEMBUKA (R)

In [ ]:
%%R
proj <- "/content/drive/MyDrive/bibliometrik-cxr"
lib  <- file.path(proj, "Rlib")
dir.create(lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib, .libPaths()))
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
  paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c("bibliometrix", "UpSetR", "readxl", "writexl", "ggplot2", "htmlwidgets")
baru <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(baru)) install.packages(baru, lib = lib)
setwd(proj)
suppressPackageStartupMessages(library(bibliometrix))
cat("Folder kerja:", getwd(), "\nbibliometrix", as.character(packageVersion("bibliometrix")), "siap\n")

In [ ]:
%%R
library(readxl)
f <- "05_figures/final_tema.xlsx"
n <- read_excel(f, sheet = "Nodes")
e <- read_excel(f, sheet = "Edges")
cat("== Kolom Nodes:\n"); print(names(n))
cat("== Kolom Edges:\n"); print(names(e))
cat("\n== NODES ==\n"); print(as.data.frame(n), right = FALSE)
cat("\n== EDGES ==\n"); print(as.data.frame(e), right = FALSE)

## Sel 3 — Corpus final + perbaikan nama LNCS

In [ ]:
%%R
library(readxl); library(writexl); library(bibliometrix)
hasil <- read_excel("03_clean/screening.xlsx")
raw   <- readRDS("03_clean/raw.rds")
print(table(hasil$keputusan)); print(table(hasil$alasan))
keep  <- hasil$EID[hasil$keputusan == "I"]
clean <- raw[raw$EID %in% keep, ]
write.csv(clean, "03_clean/corpus_S1_clean.csv", row.names = FALSE, fileEncoding = "UTF-8")
write.csv(clean[clean$Year >= 2022, ], "03_clean/corpus_2022_2026.csv", row.names = FALSE, fileEncoding = "UTF-8")
write.csv(hasil[hasil$keputusan == "I", c("EID", "Title", "Year", "tugas")], "03_clean/tugas.csv",
          row.names = FALSE, fileEncoding = "UTF-8")
M <- convert2df("03_clean/corpus_S1_clean.csv", dbsource = "scopus", format = "csv")
# Perbaikan: satukan varian nama Lecture Notes in Computer Science
M$SO <- ifelse(grepl("^LECTURE NOTES IN COMPUTER SCIENCE", M$SO), "LECTURE NOTES IN COMPUTER SCIENCE", M$SO)
saveRDS(M, "03_clean/M.rds")
cat("Corpus final:", nrow(M), "| 2022-2026:", sum(clean$Year >= 2022), "\n")

In [ ]:
%%R
gold <- toupper(c("10.xxxx/...", "10.xxxx/..."))   # isi 12 DOI gold set
M <- readRDS("03_clean/M.rds")
print(M$TI[toupper(M$DI) %in% gold])

## Sel 4 — Analisis performa

In [ ]:
%%R
library(bibliometrix); library(writexl); library(ggplot2)
M <- readRDS("03_clean/M.rds")
res <- biblioAnalysis(M, sep = ";")
S   <- summary(res, k = 20, pause = FALSE, verbose = FALSE)
lc  <- localCitations(M, sep = ";")
br  <- bradford(M)
write_xlsx(list(Info = S$MainInformationDF, Tahunan = S$AnnualProduction,
  Penulis = S$MostProdAuthors, Sitasi = S$MostCitedPapers, LocalCited = lc$Papers,
  Negara = S$MostProdCountries, Sumber = S$MostRelSources, Bradford = br$table),
  "05_figures/final_performa.xlsx")
p <- plot(res, k = 10, pause = FALSE)
for (nm in names(p)) ggsave(sprintf("05_figures/final_performa_%s.png", nm), p[[nm]], width = 8, height = 5, dpi = 300)
ggsave("05_figures/final_performa_bradford.png", br$graph, width = 8, height = 5, dpi = 300)
ann <- table(M$PY); print(ann)
t0 <- min(as.numeric(names(ann)))
cagr <- (ann[["2025"]] / ann[[as.character(t0)]])^(1 / (2025 - t0)) - 1
cat("CAGR", t0, "-2025:", round(cagr * 100, 1), "%\n")
print(S$MainInformationDF)
print(head(br$table, 10))

## Sel 5 — Cek COVID, thematic map, thematic evolution (sinonim CAD diperbaiki)

In [ ]:
%%R
library(bibliometrix); library(writexl); library(ggplot2)
M <- readRDS("03_clean/M.rds")
sin <- c(
  "PSEUDO-LABELING;PSEUDO LABELING;PSEUDO-LABELLING;PSEUDO LABEL;PSEUDO-LABEL;PSEUDO LABELS;PSEUDO-LABELS",
  "CHEST X-RAY;CHEST X-RAYS;CXR;CHEST X-RAY (CXR);CHEST RADIOGRAPH;CHEST RADIOGRAPHY;CHEST RADIOGRAPHS;CHEST X-RAY IMAGES;CHEST X-RAY IMAGE",
  "SEMI-SUPERVISED LEARNING;SEMI SUPERVISED LEARNING;SEMISUPERVISED LEARNING;SSL;SEMI-SUPERVISED;SEMI-SUPERVISED DEEP LEARNING;SEMI-SUPERVISED CLASSIFICATION",
  "CLASS IMBALANCE;IMBALANCED DATA;DATA IMBALANCE;IMBALANCED LEARNING;IMBALANCED DATASET;CLASS IMBALANCE CLASSIFICATION",
  "MULTI-LABEL CLASSIFICATION;MULTILABEL CLASSIFICATION;MULTI LABEL CLASSIFICATION;MULTI-LABEL",
  "COVID-19;COVID 19;COVID;SARS-COV-2;CORONAVIRUS",
  "MEDICAL IMAGE SEGMENTATION;SEGMENTATION;IMAGE SEGMENTATION",
  "MEDICAL IMAGE CLASSIFICATION;IMAGE CLASSIFICATION;CLASSIFICATION;CHEST X-RAY CLASSIFICATION",
  "COMPUTER AIDED DIAGNOSIS;COMPUTER-AIDED DIAGNOSIS")
hapus <- c("DEEP LEARNING", "MACHINE LEARNING", "ARTIFICIAL INTELLIGENCE",
           "SEMI-SUPERVISED LEARNING", "CHEST X-RAY", "MEDICAL IMAGING", "MEDICAL IMAGE")
txt <- tolower(paste(M$TI, M$AB, M$DE))
covid <- grepl("covid|sars-cov|coronavirus", txt)
print(table(Tahun = M$PY, COVID = ifelse(covid, "covid", "non_covid")))
tm <- thematicMap(M, field = "DE", n = 250, minfreq = 3, stemming = FALSE, size = 0.5, n.labels = 3,
                  repel = TRUE, synonyms = sin, remove.terms = hapus)
ggsave("05_figures/final_thematic_map.png", tm$map, width = 9, height = 7, dpi = 300)
te <- thematicEvolution(M, field = "DE", years = c(2020, 2022), n = 250, minFreq = 2,
                        synonyms = sin, remove.terms = hapus)
write_xlsx(list(Klaster = tm$clusters, Kata = tm$words, Nodes = te$Nodes, Edges = te$Edges),
           "05_figures/final_tema.xlsx")
print(tm$clusters[, c("groups", "name_full", "n", "centrality", "density")], width = 200)

In [ ]:
%%R
library(ggplot2)
cl <- tm$clusters
b  <- ggplot_build(tm$map)
for (d in b$data) {
  if ("xintercept" %in% names(d)) cat("Garis vertikal (x):", d$xintercept[1], "\n")
  if ("yintercept" %in% names(d)) cat("Garis horizontal (y):", d$yintercept[1], "\n")
}
kol <- intersect(c("groups", "name", "centrality", "density", "rcentrality", "rdensity"), names(cl))
print(cl[, kol], width = 200)

In [ ]:
%%R
w <- tm$words
print(names(w))
print(w[grepl("imbalanc|long-tail|long tail|minority|rare", w$Words, ignore.case = TRUE), ], width = 200)
print(w[grepl("multi-label", w$Cluster_Label, ignore.case = TRUE), ], width = 200)

## Sel 6 — Irisan dan UpSet

In [ ]:
%%R
library(UpSetR); library(writexl)
M <- readRDS("03_clean/M.rds")
txt <- tolower(paste(M$TI, M$AB, M$DE, M$ID))
f <- data.frame(
  MultiLabel = as.integer(grepl("multi-?label|multi label|single positive|partial label", txt)),
  Imbalance  = as.integer(grepl("imbalanc|long-?tail|long tail|rare disease|minority class", txt)),
  Shift      = as.integer(grepl("continual|lifelong|incremental learning|domain shift|distribution shift|label shift|prior shift", txt)))
print(colSums(f))
cat("L3:", sum(f$MultiLabel & f$Imbalance), "| L5:", sum(f$Imbalance & f$Shift), "| di luar semua lensa:", sum(rowSums(f) == 0), "\n")
png("05_figures/final_upset_irisan.png", width = 2400, height = 1500, res = 300)
print(upset(f, sets = c("MultiLabel", "Imbalance", "Shift"), order.by = "freq", keep.order = TRUE))
invisible(dev.off())
pilih <- f$MultiLabel == 1 | f$Imbalance == 1
write_xlsx(data.frame(judul = M$TI[pilih], tahun = M$PY[pilih], L1 = f$MultiLabel[pilih], L2 = f$Imbalance[pilih]),
           "03_clean/final_daftar_L1L2.xlsx")   # matriks_ekstraksi.xlsx TIDAK ditimpa

In [ ]:
   %%R
   print(M$TI[f$Shift == 1])

## Sel 7 — Burst (utama dan sensitivitas)

In [ ]:
%%R
library(writexl)
M <- readRDS("03_clean/M.rds")
sin <- c(
  "PSEUDO-LABELING;PSEUDO LABELING;PSEUDO-LABELLING;PSEUDO LABEL;PSEUDO-LABEL;PSEUDO LABELS;PSEUDO-LABELS",
  "CHEST X-RAY;CHEST X-RAYS;CXR;CHEST X-RAY (CXR);CHEST RADIOGRAPH;CHEST RADIOGRAPHY;CHEST RADIOGRAPHS;CHEST X-RAY IMAGES;CHEST X-RAY IMAGE",
  "SEMI-SUPERVISED LEARNING;SEMI SUPERVISED LEARNING;SEMISUPERVISED LEARNING;SSL;SEMI-SUPERVISED;SEMI-SUPERVISED DEEP LEARNING;SEMI-SUPERVISED CLASSIFICATION",
  "CLASS IMBALANCE;IMBALANCED DATA;DATA IMBALANCE;IMBALANCED LEARNING;IMBALANCED DATASET;CLASS IMBALANCE CLASSIFICATION",
  "MULTI-LABEL CLASSIFICATION;MULTILABEL CLASSIFICATION;MULTI LABEL CLASSIFICATION;MULTI-LABEL",
  "COVID-19;COVID 19;COVID;SARS-COV-2;CORONAVIRUS",
  "MEDICAL IMAGE SEGMENTATION;SEGMENTATION;IMAGE SEGMENTATION",
  "MEDICAL IMAGE CLASSIFICATION;IMAGE CLASSIFICATION;CLASSIFICATION;CHEST X-RAY CLASSIFICATION",
  "COMPUTER AIDED DIAGNOSIS;COMPUTER-AIDED DIAGNOSIS")
hapus <- c("DEEP LEARNING", "MACHINE LEARNING", "ARTIFICIAL INTELLIGENCE",
           "SEMI-SUPERVISED LEARNING", "CHEST X-RAY", "MEDICAL IMAGING", "MEDICAL IMAGE")
norm_kw <- function(x) {
  if (is.na(x)) return(character(0))
  k <- toupper(trimws(unlist(strsplit(x, ";")))); k <- k[k != ""]
  for (s in sin) { v <- strsplit(s, ";")[[1]]; k[k %in% v] <- v[1] }
  setdiff(unique(k), hapus)
}
kw <- lapply(M$DE, norm_kw); years <- 2018:2026
d <- as.numeric(table(factor(M$PY, levels = years))); freq <- table(unlist(kw))
burst_states <- function(r, d, s = 2, gamma = 1) {
  n <- length(r); p0 <- sum(r) / sum(d); p1 <- min(s * p0, 0.9999)
  cost <- function(p) -(r * log(p) + (d - r) * log(1 - p))
  c0 <- cost(p0); c1 <- cost(p1); tau <- gamma * log(n)
  C <- matrix(Inf, 2, n); B <- matrix(0L, 2, n); C[1, 1] <- c0[1]; C[2, 1] <- c1[1] + tau
  for (t in 2:n) { a <- C[1, t - 1]; b <- C[2, t - 1]
    if (a <= b) { C[1, t] <- a + c0[t]; B[1, t] <- 1L } else { C[1, t] <- b + c0[t]; B[1, t] <- 2L }
    if (a + tau <= b) { C[2, t] <- a + tau + c1[t]; B[2, t] <- 1L } else { C[2, t] <- b + c1[t]; B[2, t] <- 2L } }
  st <- integer(n); st[n] <- which.min(C[, n]); for (t in n:2) st[t - 1] <- B[st[t], t]
  list(st = st, gain = c0 - c1)
}
run_burst <- function(min_freq, gamma) {
  out <- list()
  for (tm in names(freq[freq >= min_freq])) {
    r <- sapply(years, function(y) sum(M$PY == y & sapply(kw, function(k) tm %in% k)))
    b <- burst_states(r, d, gamma = gamma)
    runs <- rle(b$st == 2); ends <- cumsum(runs$lengths); starts <- ends - runs$lengths + 1
    for (i in which(runs$values)) out[[length(out) + 1]] <- data.frame(istilah = tolower(tm), frekuensi = sum(r),
      mulai = years[starts[i]], selesai = years[ends[i]], kekuatan = round(sum(b$gain[starts[i]:ends[i]]), 2))
  }
  bt <- do.call(rbind, out); bt[order(bt$mulai, -bt$kekuatan), ]
}
utama <- run_burst(3, 1); sensitivitas <- run_burst(2, 0.5)
cat("== Burst utama ==\n"); print(utama, row.names = FALSE)
cat("== Burst sensitivitas ==\n"); print(sensitivitas, row.names = FALSE)
write_xlsx(list(utama = utama, sensitivitas = sensitivitas), "05_figures/final_burst.xlsx")

In [ ]:
%%R
M <- readRDS("03_clean/M.rds")
txt <- tolower(paste(M$TI, M$AB, M$DE, M$ID))
shift2 <- grepl("domain shift|distribution shift|label shift|prior shift|distribution mismatch|domain adaptation|domain generali|open[- ]set|out-of-distribution", txt)
imb <- grepl("imbalanc|long-?tail|long tail|rare disease|minority class", txt)
cat("Shift diperluas:", sum(shift2), "| Imbalance x shift:", sum(shift2 & imb), "\n")
print(M$TI[shift2 & imb])